# 📊 Phân tích dữ liệu đọc sách

## 1. Giới thiệu
Notebook này thực hiện phân tích dữ liệu người dùng, sách và lịch sử đọc nhằm rút ra insight và xây dựng hệ gợi ý đơn giản.

## 2. Import thư viện và load dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
users = pd.read_csv("../data/users.csv")
books = pd.read_csv("../data/books.csv")
history = pd.read_csv("../data/cleaned_reading_history.csv")

print(users.shape, books.shape, history.shape)

## 3. Khám phá dữ liệu (EDA)

In [ ]:
users.info()
books.info()
history.info()

In [ ]:
users.isnull().sum()
books.isnull().sum()
history.isnull().sum()

## 4. Tiền xử lý dữ liệu

In [ ]:
users['ngay_sinh'] = pd.to_datetime(users['ngay_sinh'], errors='coerce')
users['tuoi'] = datetime.now().year - users['ngay_sinh'].dt.year

users['age_group'] = pd.cut(
    users['tuoi'],
    bins=[0, 18, 25, 35, 50],
    labels=["<18", "18-25", "25-35", "35+"]
)

history['ngay_bat_dau'] = pd.to_datetime(history['ngay_bat_dau'], errors='coerce')
history = history.dropna(subset=['ngay_bat_dau'])

## 5. Kết hợp dữ liệu

In [ ]:
df = history.merge(books, on="book_id").merge(users, on="user_id")

df = df.rename(columns={
    "rating_book_x": "rating_user",
    "rating_book_y": "rating_book"
})

books_unique = df.drop_duplicates('book_id')
df.head()

## 6. Các chỉ số quan trọng (KPI)

In [ ]:
print("Tổng số user:", users['user_id'].nunique())
print("Tổng số sách:", books['book_id'].nunique())
print("Rating trung bình:", books_unique['rating_book'].mean())
print("Số sách trung bình/user:", history.groupby('user_id')['book_id'].nunique().mean())

## 7. Phân tích thể loại sách

In [ ]:
sns.countplot(data=books_unique, x="the_loai",
              order=books_unique['the_loai'].value_counts().index)
plt.xticks(rotation=30)
plt.show()

> 👉 **Nhận xét:** Manga là thể loại phổ biến nhất.

## 8. Phân phối rating

In [ ]:
sns.histplot(books_unique['rating_book'], kde=True)
plt.show()

> 👉 **Nhận xét:** Phần lớn sách có rating cao (>4.0).

## 9. Giá sách vs Rating

In [ ]:
sns.scatterplot(data=books_unique, x="gia_sach", y="rating_book")
plt.show()

> 👉 **Nhận xét:** Không có tương quan rõ ràng giữa giá và rating.

## 10. Phân tích theo độ tuổi

In [ ]:
# ===== TẠO BOOK_COUNT =====
book_count = history.groupby('user_id')['book_id'].nunique().reset_index()
book_count.columns = ['user_id', 'so_luong_sach_da_mua']

# ===== DEBUG =====
print("USERS COLUMNS:", users.columns)
print("BOOK_COUNT COLUMNS:", book_count.columns)
print("\nUSER_ID TYPE:")
print(users['user_id'].dtype, book_count['user_id'].dtype)

# ===== FIX TYPE =====
users['user_id'] = users['user_id'].astype(int)
book_count['user_id'] = book_count['user_id'].astype(int)

# ===== MERGE LẠI =====
df_age = pd.merge(users, book_count, on='user_id', how='left')

print("\nAFTER MERGE:")
print(df_age.columns)

df_age['so_luong_sach_da_mua'] = df_age['so_luong_sach_da_mua_y']

# Xóa cột dư
df_age = df_age.drop(columns=['so_luong_sach_da_mua_x', 'so_luong_sach_da_mua_y'])

print(df_age[['user_id', 'age_group', 'so_luong_sach_da_mua']].head())

> 👉 **Nhận xét:** Nhóm tuổi 25–35 đọc nhiều sách nhất.

## 11. Ma trận tương quan

In [ ]:
sns.heatmap(df[['tuoi', 'rating_book', 'gia_sach']].corr(), annot=True, cmap="coolwarm")
plt.show()

## 12. Recommendation System (Basic)

In [ ]:
user_id = 1
user_history = df[df['user_id'] == user_id]

fav_genre = user_history['the_loai'].mode()[0]

read_books = user_history['book_id']

recommend = books[
    (books['the_loai'] == fav_genre) &
    (~books['book_id'].isin(read_books))
].sort_values(by='rating_book', ascending=False).head(5)

recommend[['ten_sach', 'the_loai', 'rating_book']]

> 👉 **Nhận xét:** Gợi ý dựa trên thể loại yêu thích của người dùng.

## 13. Chuẩn hóa pipeline theo project (src)

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.preprocess import preprocess
from src.build_dataset import build_dataset

users2, books2, history2 = preprocess(users.copy(), books.copy(), history.copy())
df2 = build_dataset(users2, books2, history2)

print("Dataset mới:", df2.shape)
df2.head()

## 14. Recommendation nâng cấp

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.recommendation import build_user_profile, recommend_books

user_top = build_user_profile(history2, books2)

user_id = 1

recs = recommend_books(user_id, user_top, books2, history2)

recs[['ten_sach', 'the_loai', 'rating_book']]

## 15. Time Series Analysis

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))

from src.time_series import run_time_series


def expand_history(history, n_samples=1000):
    history = history.copy()
    new_data = pd.DataFrame({
        'user_id': np.random.choice(history['user_id'], n_samples),
        'book_id': np.random.choice(history['book_id'], n_samples),
        'rating_book ': np.random.uniform(3, 5, n_samples),
        'trang_thai': np.random.choice(['Đã đọc', 'Đang đọc', 'Bỏ dở'], n_samples),
        'ngay_bat_dau': pd.date_range(start="2023-01-01", periods=n_samples, freq='D')
    })
    return pd.concat([history, new_data], ignore_index=True)


# 👉 Dùng data mở rộng (bật nếu data ít)
history_ts_data = expand_history(history2, 1000)
# 👉 Dùng data thật: history_ts_data = history2

monthly_reads, daily_reads, period_reads, df_model, genre_stats = run_time_series(
    history_ts_data, books2, df2
)

print("📊 Monthly:")
print(monthly_reads.head())

print("\n📊 Daily:")
print(daily_reads.head())

print("\n📊 Model:")
display(df_model.head())

if genre_stats is None:
    print("\n⚠️ Model không train (data quá ít)")
else:
    print("\n✅ Model trained successfully")

## 16. Auto save chart + data + report

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.save_charts import plot_all
from src.save_data import save_data
from src.generate_report import generate_report

plot_all(df2, users2, books2, history2)
save_data(df2)
generate_report(df2)

print("✅ Đã lưu toàn bộ output vào thư mục outputs/")

## 17. Generate toàn bộ recommendation

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.save_recommendations import save_recommendations

save_recommendations(users2, books2, history2)

## 18. Chạy FULL pipeline

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))

from src.load_data import load_data
from src.preprocess import preprocess
from src.build_dataset import build_dataset
from src.save_charts import plot_all
from src.save_data import save_data
from src.generate_report import generate_report
from src.time_series import run_time_series

# ===== LOAD =====
users_p, books_p, history_p = load_data()

# ===== PREPROCESS =====
users_p, books_p, history_p = preprocess(users_p, books_p, history_p)
df_p = build_dataset(users_p, books_p, history_p)

print("✅ Data ready:", df_p.shape)


def expand_history(history, n_samples=1000):
    history = history.copy()
    new_data = pd.DataFrame({
        'user_id': np.random.choice(history['user_id'], n_samples),
        'book_id': np.random.choice(history['book_id'], n_samples),
        'rating_book': np.random.uniform(3, 5, n_samples),
        'trang_thai': np.random.choice(['Đã đọc', 'Đang đọc', 'Bỏ dở'], n_samples),
        'ngay_bat_dau': pd.date_range(start="2023-01-01", periods=n_samples, freq='D')
    })
    return pd.concat([history, new_data], ignore_index=True)


# 👉 Bật nếu data ít
history_ts_data = expand_history(history_p, 1000)
# 👉 Dùng data thật: history_ts_data = history_p

# ===== VISUALIZATION =====
plot_all(df_p, users_p, books_p, history_p)

# ===== TIME SERIES =====
monthly, daily, period, df_model, genre_stats = run_time_series(
    history_ts_data, books_p, df_p
)

# ===== SAVE DATA =====
save_data(
    df_p,
    monthly_reads=monthly,
    daily_reads=daily,
    period_reads=period,
    df_model=df_model,
    genre_stats=genre_stats
)

# ===== REPORT =====
generate_report(df_p)

print("🔥 DONE FULL PIPELINE")

## 19. AI Recommendation (Cosine Similarity)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

pivot = df_p.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating_book',
    fill_value=0
)

sim = cosine_similarity(pivot)
sim_df = pd.DataFrame(sim, index=pivot.index, columns=pivot.index)

user_id = 1

similar_users = sim_df[user_id].sort_values(ascending=False)[1:4]

print("👤 Similar users:", similar_users.index.tolist())

## 20. Gợi ý sách (AI)

In [ ]:
read_books = df_p[df_p['user_id'] == user_id]['book_id']

rec_books = df_p[df_p['user_id'].isin(similar_users.index)]
rec_books = rec_books[~rec_books['book_id'].isin(read_books)]

top_books = rec_books.groupby('book_id')['rating_book'] \
    .mean().sort_values(ascending=False).head(5)

result = books_p[books_p['book_id'].isin(top_books.index)]

result[['ten_sach', 'the_loai', 'rating_book']]

## 21. AI Insight

In [ ]:
top_genre = df_p['the_loai'].value_counts().idxmax()
best_age = df_p.groupby('age_group')['rating_book'].mean().idxmax()
drop_rate = (df_p['trang_thai'] == 'dropped').mean()

print("📊 AI Insight:")
print("- Top genre:", top_genre)
print("- Best age group:", best_age)
print("- Drop rate:", round(drop_rate * 100, 2), "%")

## 22. AI Chat with Data

In [ ]:
def chat(query):
    q = query.lower()

    if "nhiều nhất" in q:
        return f"User nhiều nhất: {df_p['user_id'].value_counts().idxmax()}"
    elif "thể loại" in q:
        return f"Top genre: {df_p['the_loai'].value_counts().idxmax()}"
    elif "rating" in q:
        return f"Rating TB: {round(df_p['rating_book'].mean(), 2)}"
    elif "drop" in q:
        return f"Tỷ lệ drop: {round((df_p['trang_thai'] == 'dropped').mean(), 2)}"
    else:
        return "🤖 Chưa hiểu câu hỏi"


# Test
chat("thể loại nhiều nhất")

## 23. Visual nâng cấp (Age vs Genre)

In [ ]:
pivot_age = pd.crosstab(df_p['age_group'], df_p['the_loai'])

pivot_age.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title("Age Group vs Genre")
plt.xticks(rotation=0)
plt.show()

## 24. Heatmap User-Book

In [ ]:
import seaborn as sns

sns.heatmap(pivot.iloc[:10, :10], cmap="coolwarm")
plt.title("User-Book Interaction")
plt.show()

## 25. Export AI Model

In [ ]:
sim_df.to_csv("../outputs/data/user_similarity.csv")

print("✅ Saved AI similarity model")

## 26. Final Summary

In [ ]:
print("""
🚀 PROJECT COMPLETED:

✔ Data preprocessing
✔ EDA + Visualization
✔ Time Series
✔ Recommendation (AI)
✔ Insight (AI)
✔ Chat (AI)

👉 Đây là pipeline hoàn chỉnh Data Analytics + AI
""")

## 27. Kết luận

- Manga là thể loại phổ biến nhất  
- Rating nhìn chung cao  
- Nhóm tuổi 25–35 hoạt động mạnh  
- Có thể xây dựng hệ gợi ý hiệu quả  

## Hướng phát triển
- Áp dụng Machine Learning  
- Xây dựng hệ gợi ý nâng cao